# ACS Charging-Demand and Community-Access Data

## Goal

Download 2020–2024 American Community Survey block-group estimates for the seven-county study area, join them to matching 2024 TIGER/Line block-group boundaries, and calculate transparent preliminary indicators for market demand and public-charging access need.

Official source: https://www.census.gov/data/developers/data-sets/acs-5year.html

## Setup

This notebook uses the Census Bureau's official table-based ACS Summary Files, so no API key is required. The source files contain the same published Detailed Table estimates used by the Census API. Only Georgia block groups in the seven-county study area are retained.

The 2024 TIGER/Line geography vintage is used to match the 2020–2024 ACS release.

In [1]:
from pathlib import Path
from urllib.request import Request, urlopen

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability")
CENSUS_DIR = PROJECT_ROOT / "data" / "raw" / "census"
BOUNDARY_DIR = PROJECT_ROOT / "data" / "raw" / "boundaries"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPATIAL_DIR = PROJECT_ROOT / "outputs" / "spatial"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
for folder in [CENSUS_DIR, BOUNDARY_DIR, PROCESSED_DIR, SPATIAL_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

COUNTIES = {
    "013": "Barrow", "059": "Clarke", "135": "Gwinnett", "139": "Hall",
    "157": "Jackson", "219": "Oconee", "297": "Walton",
}
VARIABLES = {
    "B01003_001E": "population",
    "B11001_001E": "households",
    "B19013_001E": "median_household_income",
    "B25003_001E": "occupied_housing_units",
    "B25003_003E": "renter_occupied_units",
    "B25024_001E": "housing_units",
    "B25024_004E": "units_3_to_4",
    "B25024_005E": "units_5_to_9",
    "B25024_006E": "units_10_to_19",
    "B25024_007E": "units_20_to_49",
    "B25024_008E": "units_50_plus",
    "B08201_001E": "vehicle_households",
    "B08201_002E": "zero_vehicle_households",
}
SUMMARY_BASE = (
    "https://www2.census.gov/programs-surveys/acs/summary_file/2024/"
    "table-based-SF/data/5YRData"
)
TIGER_URL = "https://www2.census.gov/geo/tiger/TIGER2024/BG/tl_2024_13_bg.zip"
boundary_zip = BOUNDARY_DIR / "tl_2024_13_bg.zip"


## Steps

### 1. Download official ACS Summary File tables

The files are cached locally after the first run. Reading them in chunks keeps memory use bounded, and the notebook immediately filters to the seven study counties.

In [2]:
def summary_column(api_variable):
    table, item = api_variable.split("_")[:2]
    return f"{table}_E{item.removesuffix('E')}"


table_variables = {}
for api_variable in VARIABLES:
    table = api_variable.split("_")[0]
    table_variables.setdefault(table, []).append(api_variable)

study_prefixes = tuple(f"1500000US13{county}" for county in COUNTIES)
study_tract_prefixes = tuple(f"1400000US13{county}" for county in COUNTIES)
table_frames = []
tract_frame = None

for table, api_variables in table_variables.items():
    filename = f"acsdt5y2024-{table.lower()}.dat"
    local_path = CENSUS_DIR / filename
    source_url = f"{SUMMARY_BASE}/{filename}"

    if not local_path.exists():
        print(f"Downloading {table}...")
        request = Request(source_url, headers={"User-Agent": "GIS-Portfolio-EV-Suitability/1.0"})
        with urlopen(request, timeout=300) as response, local_path.open("wb") as output:
            while chunk := response.read(1024 * 1024):
                output.write(chunk)

    source_columns = ["GEO_ID"] + [summary_column(variable) for variable in api_variables]
    selected_chunks = []
    for chunk in pd.read_csv(
        local_path,
        sep="|",
        usecols=source_columns,
        dtype={"GEO_ID": "string"},
        chunksize=200_000,
        low_memory=False,
    ):
        prefixes = study_tract_prefixes if table == "B08201" else study_prefixes
        selected = chunk.loc[chunk["GEO_ID"].str.startswith(prefixes, na=False)].copy()
        if not selected.empty:
            selected_chunks.append(selected)

    table_frame = pd.concat(selected_chunks, ignore_index=True)
    table_frame = table_frame.rename(
        columns={summary_column(variable): variable for variable in api_variables}
    )
    if table == "B08201":
        table_frame["TRACT_GEOID"] = table_frame["GEO_ID"].str.split("US").str[-1]
        tract_frame = table_frame.drop(columns="GEO_ID")
        geography_label = "tracts"
    else:
        table_frames.append(table_frame)
        geography_label = "block groups"
    print(f"{table}: retained {len(table_frame):,} study-area {geography_label}")

acs = table_frames[0]
for table_frame in table_frames[1:]:
    acs = acs.merge(table_frame, on="GEO_ID", how="outer", validate="one_to_one")

acs["GEOID"] = acs["GEO_ID"].str.split("US").str[-1]
acs["TRACT_GEOID"] = acs["GEOID"].str[:11]
acs = acs.merge(tract_frame, on="TRACT_GEOID", how="left", validate="many_to_one")
acs["state"] = acs["GEOID"].str[:2]
acs["county"] = acs["GEOID"].str[2:5]
acs["tract"] = acs["GEOID"].str[5:11]
acs["block group"] = acs["GEOID"].str[11:]
acs["county_name"] = acs["county"].map(COUNTIES)
acs["NAME"] = (
    "Block Group " + acs["block group"] + ", Census Tract " + acs["tract"]
    + ", " + acs["county_name"] + " County, Georgia"
)

acs = acs.rename(columns=VARIABLES)
numeric_fields = list(VARIABLES.values())
acs[numeric_fields] = acs[numeric_fields].apply(pd.to_numeric, errors="coerce")
acs.loc[acs["median_household_income"] < 0, "median_household_income"] = np.nan

raw_path = CENSUS_DIR / "acs_2020_2024_ev_demand_block_groups.csv"
acs.to_csv(raw_path, index=False)
print(f"Loaded {len(acs):,} block groups across {acs['county'].nunique()} counties.")
print("Saved:", raw_path)

B01003: retained 926 study-area block groups


B11001: retained 926 study-area block groups


B19013: retained 926 study-area block groups


B25003: retained 926 study-area block groups


B25024: retained 926 study-area block groups


B08201: retained 370 study-area tracts


Loaded 926 block groups across 7 counties.
Saved: C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability\data\raw\census\acs_2020_2024_ev_demand_block_groups.csv


### 2. Join matching TIGER/Line block-group boundaries

In [3]:
if not boundary_zip.exists():
    request = Request(TIGER_URL, headers={"User-Agent": "GIS-Portfolio-EV-Suitability/1.0"})
    with urlopen(request, timeout=120) as response:
        boundary_zip.write_bytes(response.read())

georgia_block_groups = gpd.read_file(f"zip://{boundary_zip}")
study_boundaries = georgia_block_groups.loc[georgia_block_groups["COUNTYFP"].isin(COUNTIES)].copy()
demand = study_boundaries.merge(acs, on="GEOID", how="left", validate="one_to_one")

assert len(demand) == len(acs), "ACS rows and study-area boundaries did not match one-to-one."
assert demand["population"].notna().all(), "Population is missing after the geographic join."

# EPSG:5070 is an equal-area CRS appropriate for density calculations in the contiguous U.S.
demand_equal_area = demand.to_crs("EPSG:5070")
demand["land_sq_miles"] = demand_equal_area.geometry.area / 2_589_988.110336
demand["population_density"] = demand["population"] / demand["land_sq_miles"].replace(0, np.nan)
demand["household_density"] = demand["households"] / demand["land_sq_miles"].replace(0, np.nan)

demand.to_file(SPATIAL_DIR / "acs_ev_demand_block_groups.gpkg", layer="ev_demand", driver="GPKG")

### 3. Calculate demand and public-access indicators

These are preliminary relative indices, not estimates of required ports. Income represents near-term market demand; renter and multifamily shares represent dependence on public charging. Keeping those concepts separate prevents the model from directing every investment toward affluent areas.

In [4]:
demand["multifamily_3_plus_units"] = demand[
    ["units_3_to_4", "units_5_to_9", "units_10_to_19", "units_20_to_49", "units_50_plus"]
].sum(axis=1, min_count=1)
demand["renter_share"] = demand["renter_occupied_units"] / demand["occupied_housing_units"].replace(0, np.nan)
demand["multifamily_share"] = demand["multifamily_3_plus_units"] / demand["housing_units"].replace(0, np.nan)
demand["zero_vehicle_share"] = demand["zero_vehicle_households"] / demand["vehicle_households"].replace(0, np.nan)

def percentile(series):
    return series.rank(pct=True, method="average")

demand["market_demand_index"] = 100 * (
    0.45 * percentile(demand["population_density"]) +
    0.30 * percentile(demand["household_density"]) +
    0.25 * percentile(demand["median_household_income"])
)
demand["public_access_need_index"] = 100 * (
    0.40 * percentile(demand["renter_share"]) +
    0.35 * percentile(demand["multifamily_share"]) +
    0.25 * percentile(demand["population_density"])
)

# Refresh the spatial output so it includes all derived indicators.
demand.to_file(SPATIAL_DIR / "acs_ev_demand_block_groups.gpkg", layer="ev_demand", driver="GPKG")

output_fields = [
    "GEOID", "county_name", "population", "households", "median_household_income",
    "population_density", "household_density", "renter_share", "multifamily_share",
    "zero_vehicle_share", "market_demand_index", "public_access_need_index",
]
table_path = PROCESSED_DIR / "acs_ev_demand_indicators.csv"
demand[output_fields].to_csv(table_path, index=False)
display(demand[output_fields].describe().round(2))
print("Saved:", table_path)

,population,households,median_household_income,population_density,household_density,renter_share,multifamily_share,zero_vehicle_share,market_demand_index,public_access_need_index
count,926.00,926.00,873.00,926.00,926.00,926.00,926.00,926.00,873.00,926.00
mean,1775.04,610.14,92850.56,2576.45,917.87,0.32,0.14,0.04,49.93,50.05
std,778.55,258.62,38283.31,2968.47,1147.22,0.29,0.24,0.06,21.49,22.98
min,334.00,71.00,6914.00,44.67,15.18,0.00,0.00,0.00,5.19,10.33
25%,1204.25,412.25,66250.00,799.25,271.12,0.10,0.00,0.01,31.87,31.96
50%,1679.50,575.00,88397.00,1841.39,629.69,0.22,0.00,0.03,51.55,44.82
75%,2185.75,772.50,111830.00,3307.21,1084.45,0.48,0.20,0.05,68.61,68.27
max,6338.00,1733.00,250001.00,36769.45,12053.18,1.00,1.00,0.40,92.31,99.39


Saved: C:\Users\cason\GIS_Portfolio_3\Project 4 - EV Charging Suitability\data\processed\acs_ev_demand_indicators.csv


## Checks

In [5]:
assert acs["GEOID"].is_unique
assert set(acs["county"]) == set(COUNTIES)
assert demand.geometry.notna().all()
assert demand["market_demand_index"].dropna().between(0, 100).all()
assert demand["public_access_need_index"].dropna().between(0, 100).all()
assert (SPATIAL_DIR / "acs_ev_demand_block_groups.gpkg").exists()
assert (PROCESSED_DIR / "acs_ev_demand_indicators.csv").exists()

print("All ACS, geography, and indicator checks passed.")
print("Block groups:", f"{len(demand):,}")
print("Counties:", demand["county_name"].nunique())

All ACS, geography, and indicator checks passed.
Block groups: 926
Counties: 7


## Next Steps

Map market demand and public-access need beside existing charging coverage. The following stage will calculate charging gaps and underserved block groups before introducing roads, traffic, and candidate land.